# 🦕 DINO SDK - Setup do Workspace Databricks

Este notebook ajuda a configurar o workspace Databricks para usar o DINO SDK, incluindo o upload do notebook core de ingestão.

## 🎯 Objetivos
- ✅ Fazer upload do notebook `dino_ingestion_core` para o workspace
- ✅ Verificar se a estrutura de pastas está correta
- ✅ Testar conectividade com o workspace
- ✅ Validar permissões necessárias

## 📋 Pré-requisitos
- Databricks CLI configurado ou variáveis de ambiente
- Permissões para criar/modificar notebooks no workspace

In [ ]:
# 1. Configuração e Verificação de Conectividade
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ImportFormat, Language
import os
from pathlib import Path

# Inicializar cliente Databricks
try:
    client = WorkspaceClient()
    current_user = client.current_user.me()
    print(f"✅ Conectado ao Databricks como: {current_user.user_name}")
    print(f"🏢 Workspace: {client.config.host}")
except Exception as e:
    print(f"❌ Erro na conexão com Databricks: {e}")
    print("🔧 Verifique suas credenciais e configuração")
    raise

In [ ]:
# 2. Definir Caminhos e Estrutura do Workspace
# Caminhos importantes para o DINO SDK
DINO_WORKSPACE_PATH = "/Workspace/dino"
CORE_NOTEBOOK_PATH = f"{DINO_WORKSPACE_PATH}/dino_ingestion_core"
LOCAL_NOTEBOOK_PATH = "notebooks/dino_ingestion_core.ipynb"

print("📂 ESTRUTURA DO DINO WORKSPACE:")
print(f"   Base path: {DINO_WORKSPACE_PATH}")
print(f"   Notebook core: {CORE_NOTEBOOK_PATH}")
print(f"   Arquivo local: {LOCAL_NOTEBOOK_PATH}")

# Verificar se a pasta base existe
try:
    # Tentar listar o diretório
    client.workspace.list(DINO_WORKSPACE_PATH)
    print(f"✅ Pasta {DINO_WORKSPACE_PATH} já existe")
except Exception:
    print(f"📁 Pasta {DINO_WORKSPACE_PATH} será criada")

In [ ]:
# 3. Criar Estrutura de Diretórios no Workspace
try:
    # Criar pasta principal do DINO se não existir
    client.workspace.mkdirs(DINO_WORKSPACE_PATH)
    print(f"✅ Estrutura de diretórios criada: {DINO_WORKSPACE_PATH}")
    
    # Listar conteúdo atual
    contents = client.workspace.list(DINO_WORKSPACE_PATH)
    print(f"📂 Conteúdo atual da pasta DINO:")
    for item in contents:
        item_type = "📁" if item.object_type.value == "DIRECTORY" else "📓"
        print(f"   {item_type} {item.path}")
        
except Exception as e:
    print(f"❌ Erro ao criar estrutura de diretórios: {e}")
    raise

In [ ]:
# 4. Ler o Notebook Core Local
import json

# Caminho absoluto para o notebook local
current_dir = Path.cwd()
notebook_path = current_dir / "notebooks" / "dino_ingestion_core.ipynb"

print(f"🔍 Procurando notebook em: {notebook_path}")

if notebook_path.exists():
    print("✅ Notebook encontrado!")
    
    # Ler conteúdo do notebook
    with open(notebook_path, 'r', encoding='utf-8') as f:
        notebook_content = f.read()
    
    print(f"📊 Tamanho do notebook: {len(notebook_content)} caracteres")
    
    # Validar se é um notebook válido
    try:
        notebook_json = json.loads(notebook_content)
        cell_count = len(notebook_json.get('cells', []))
        print(f"📋 Notebook válido com {cell_count} células")
    except Exception as e:
        print(f"⚠️ Aviso: Erro na validação do JSON: {e}")
        
else:
    print("❌ Notebook não encontrado no caminho local")
    print("🔧 Certifique-se de que o notebook dino_ingestion_core.ipynb existe")
    raise FileNotFoundError(f"Notebook não encontrado: {notebook_path}")

In [ ]:
# 5. Upload do Notebook para o Workspace
try:
    print("🚀 Fazendo upload do notebook core para o workspace...")
    
    # Fazer upload do notebook
    client.workspace.upload(
        path=CORE_NOTEBOOK_PATH,
        content=notebook_content.encode('utf-8'),
        format=ImportFormat.JUPYTER,
        language=Language.PYTHON,
        overwrite=True
    )
    
    print(f"✅ Upload concluído com sucesso!")
    print(f"📓 Notebook disponível em: {CORE_NOTEBOOK_PATH}")
    
    # Verificar se o upload foi bem-sucedido
    notebook_info = client.workspace.get_status(CORE_NOTEBOOK_PATH)
    print(f"📊 Informações do notebook:")
    print(f"   - Tipo: {notebook_info.object_type}")
    print(f"   - Linguagem: {notebook_info.language}")
    print(f"   - Caminho: {notebook_info.path}")
    
except Exception as e:
    print(f"❌ Erro no upload: {e}")
    raise

In [ ]:
# 6. Verificar Estrutura Final do Workspace
try:
    print("🔍 VERIFICAÇÃO FINAL DA ESTRUTURA:")
    print("=" * 50)
    
    # Listar conteúdo da pasta DINO
    dino_contents = client.workspace.list(DINO_WORKSPACE_PATH)
    
    for item in dino_contents:
        item_type = "📁" if item.object_type.value == "DIRECTORY" else "📓"
        print(f"{item_type} {item.path}")
        
        # Se for o notebook core, mostrar detalhes
        if item.path == CORE_NOTEBOOK_PATH:
            print(f"   ✅ Notebook core encontrado e pronto para uso!")
            print(f"   🔗 URL: {client.config.host}/#workspace{item.path}")
    
    print("\n🎉 SETUP DO WORKSPACE CONCLUÍDO!")
    
except Exception as e:
    print(f"❌ Erro na verificação final: {e}")

In [ ]:
# 7. Teste de Conectividade e Permissões
print("🧪 TESTE DE CONECTIVIDADE E PERMISSÕES:")
print("=" * 50)

tests_passed = 0
total_tests = 4

# Teste 1: Ler notebook
try:
    notebook_status = client.workspace.get_status(CORE_NOTEBOOK_PATH)
    print("✅ Teste 1: Leitura do notebook - PASSOU")
    tests_passed += 1
except Exception as e:
    print(f"❌ Teste 1: Leitura do notebook - FALHOU: {e}")

# Teste 2: Listar diretório
try:
    contents = client.workspace.list(DINO_WORKSPACE_PATH)
    print("✅ Teste 2: Listagem de diretório - PASSOU")
    tests_passed += 1
except Exception as e:
    print(f"❌ Teste 2: Listagem de diretório - FALHOU: {e}")

# Teste 3: Verificar permissões de escrita (tentar criar arquivo temporário)
try:
    temp_path = f"{DINO_WORKSPACE_PATH}/temp_test"
    client.workspace.upload(
        path=temp_path,
        content=b"teste",
        format=ImportFormat.AUTO,
        overwrite=True
    )
    client.workspace.delete(temp_path)  # Limpar teste
    print("✅ Teste 3: Permissões de escrita - PASSOU")
    tests_passed += 1
except Exception as e:
    print(f"❌ Teste 3: Permissões de escrita - FALHOU: {e}")

# Teste 4: Verificar acesso ao Unity Catalog
try:
    catalogs = client.catalogs.list()
    catalog_count = len(list(catalogs))
    print(f"✅ Teste 4: Acesso ao Unity Catalog - PASSOU ({catalog_count} catálogos)")
    tests_passed += 1
except Exception as e:
    print(f"❌ Teste 4: Acesso ao Unity Catalog - FALHOU: {e}")

# Resultado final
print(f"\n📊 RESULTADO DOS TESTES: {tests_passed}/{total_tests}")
if tests_passed == total_tests:
    print("🎉 TODOS OS TESTES PASSARAM! Workspace está pronto para o DINO SDK")
else:
    print("⚠️ Alguns testes falharam. Verifique permissões e configurações.")

In [ ]:
# 8. Instruções Finais e Próximos Passos
print("📋 SETUP COMPLETO - PRÓXIMOS PASSOS:")
print("=" * 40)

setup_summary = {
    "workspace_host": client.config.host,
    "dino_path": DINO_WORKSPACE_PATH,
    "core_notebook": CORE_NOTEBOOK_PATH,
    "notebook_url": f"{client.config.host}/#workspace{CORE_NOTEBOOK_PATH}",
    "setup_date": "2025-09-06"
}

for key, value in setup_summary.items():
    print(f"   {key}: {value}")

print(f"\n🚀 COMO USAR O DINO SDK:")
print("1. Instalar o pacote: pip install dino_sdk-2.0.0-py3-none-any.whl")
print("2. Criar job: create_dino_job('catalog', 'schema', 'table', True)")
print("3. O job automaticamente usará o notebook core configurado")

print(f"\n🔗 LINKS IMPORTANTES:")
print(f"📓 Notebook Core: {setup_summary['notebook_url']}")
print(f"📂 Pasta DINO: {client.config.host}/#workspace{DINO_WORKSPACE_PATH}")

print(f"\n✅ SETUP DO DINO SDK WORKSPACE FINALIZADO!")

## 📋 Resumo do Setup

### ✅ **O que foi Configurado**
1. **Estrutura de Pastas**: `/Workspace/dino/` criada
2. **Notebook Core**: `dino_ingestion_core` carregado
3. **Permissões**: Verificadas para leitura/escrita
4. **Unity Catalog**: Acesso confirmado

### 🎯 **Próximos Passos**
1. **Usar o DINO SDK**:
   ```python
   from dino_sdk import create_dino_job
   
   result = create_dino_job(
       catalog_name="meu_catalogo",
       schema_name="meu_schema", 
       table_name="minha_tabela",
       is_automated=True
   )
   ```

2. **Jobs Automáticos**: Os jobs criados usarão automaticamente o notebook `dino_ingestion_core`

3. **Monitoramento**: Acompanhe execuções via Databricks Jobs UI

### 🔧 **Troubleshooting**
- **Erro de permissão**: Verifique se o usuário tem acesso ao workspace
- **Notebook não encontrado**: Execute este setup novamente
- **Unity Catalog**: Confirme acesso aos catálogos necessários